# ALS implicit with PyTorch on 2 GPUs (chunked, no threads)

In [ ]:

import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.device_count())


In [ ]:

base = Path('/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output')
files = sorted(base.glob('test/*.parquet'))
dfs = [pq.read_table(f).to_pandas() for f in files]
inter = pd.concat(dfs, ignore_index=True)
users_raw = inter['user_idx'].to_numpy()
items_raw = inter['item_idx'].to_numpy()
ratings = inter['rating'].fillna(1.0).to_numpy(dtype=np.float32)
u_uniq, u_inv = np.unique(users_raw, return_inverse=True)
i_uniq, i_inv = np.unique(items_raw, return_inverse=True)
U = len(u_uniq); I = len(i_uniq)
print('U', U, 'I', I)
u_ids = u_inv.astype(np.int64)
i_ids = i_inv.astype(np.int64)
conf = ratings.astype(np.float32)
pref = (ratings > 0).astype(np.float32)
order = np.argsort(u_ids)
u_sorted = u_ids[order]; i_sorted = i_ids[order]; c_sorted = conf[order]; p_sorted = pref[order]
user_ptr = np.searchsorted(u_sorted, np.arange(U+1))
order_i = np.argsort(i_ids)
i_sorted_i = i_ids[order_i]; u_sorted_i = u_ids[order_i]; c_sorted_i = conf[order_i]; p_sorted_i = pref[order_i]
item_ptr = np.searchsorted(i_sorted_i, np.arange(I+1))


In [ ]:

FACTORS = 32
REG = 0.1
ITERS = 2
CHUNK_U = 2048
CHUNK_I = 4096
device0 = torch.device('cuda:0')
device1 = torch.device('cuda:1')

def init_factors(n, f, seed=42):
    g = torch.Generator().manual_seed(seed)
    return (torch.rand(n, f, generator=g) * 0.01).numpy().astype(np.float32)

X = init_factors(U, FACTORS)
Y = init_factors(I, FACTORS)

def process_user_chunk(start_u, end_u, Ymat, device):
    Uc = end_u - start_u
    A = torch.zeros(Uc, FACTORS, FACTORS, device=device)
    b = torch.zeros(Uc, FACTORS, device=device)
    A += (torch.eye(FACTORS, device=device) * REG).unsqueeze(0)
    lo = int(user_ptr[start_u]); hi = int(user_ptr[end_u])
    bs = 2048
    for s in range(lo, hi, bs):
        e = min(s+bs, hi)
        local = torch.from_numpy(u_sorted[s:e] - start_u).to(device)
        ii = i_sorted[s:e]
        cc = torch.from_numpy(c_sorted[s:e]).to(device)
        pp = torch.from_numpy(p_sorted[s:e]).to(device)
        Yi = torch.from_numpy(Ymat[ii]).to(device)
        outer = torch.einsum('mi,mj->mij', Yi, Yi) * cc[:, None, None]
        A.index_add_(0, local, outer)
        b.index_add_(0, local, Yi * (cc * pp)[:, None])
    Xc = torch.linalg.solve(A, b.unsqueeze(2)).squeeze(2)
    return Xc.detach().cpu().numpy()

def update_users(X, Y):
    i = 0
    chunk_idx = 0
    while i < U:
        j = min(i+CHUNK_U, U)
        dev = device0 if (chunk_idx % 2 == 0) else device1
        X[i:j] = process_user_chunk(i, j, Y, dev)
        i = j
        chunk_idx += 1

def process_item_chunk(start_i, end_i, Xmat, device):
    Ic = end_i - start_i
    A = torch.zeros(Ic, FACTORS, FACTORS, device=device)
    b = torch.zeros(Ic, FACTORS, device=device)
    A += (torch.eye(FACTORS, device=device) * REG).unsqueeze(0)
    lo = int(item_ptr[start_i]); hi = int(item_ptr[end_i])
    bs = 2048
    for s in range(lo, hi, bs):
        e = min(s+bs, hi)
        local = torch.from_numpy(i_sorted_i[s:e] - start_i).to(device)
        uu = u_sorted_i[s:e]
        cc = torch.from_numpy(c_sorted_i[s:e]).to(device)
        pp = torch.from_numpy(p_sorted_i[s:e]).to(device)
        Xu = torch.from_numpy(Xmat[uu]).to(device)
        outer = torch.einsum('mi,mj->mij', Xu, Xu) * cc[:, None, None]
        A.index_add_(0, local, outer)
        b.index_add_(0, local, Xu * (cc * pp)[:, None])
    Yc = torch.linalg.solve(A, b.unsqueeze(2)).squeeze(2)
    return Yc.detach().cpu().numpy()

def update_items(Y, X):
    i = 0
    chunk_idx = 0
    while i < I:
        j = min(i+CHUNK_I, I)
        dev = device0 if (chunk_idx % 2 == 0) else device1
        Y[i:j] = process_item_chunk(i, j, X, dev)
        i = j
        chunk_idx += 1

for it in range(ITERS):
    t0 = time.time()
    update_users(X, Y)
    update_items(Y, X)
    print(f'iter {it+1}/{ITERS} done in {time.time()-t0:.1f}s', flush=True)


In [ ]:

out = Path('/kaggle/working')
np.save(out/'als_torch_user_factors.npy', X)
np.save(out/'als_torch_item_factors.npy', Y)
np.save(out/'als_torch_user_ids.npy', u_uniq)
np.save(out/'als_torch_item_ids.npy', i_uniq)
import json
meta = {'U': int(U), 'I': int(I), 'factors': FACTORS, 'iters': ITERS, 'interactions': int(len(inter))}
(out/'als_torch_meta.json').write_text(json.dumps(meta))
print(json.dumps(meta, indent=2))


In [ ]:
!ls -lh /kaggle/working